# Pravaala: silver annotation of GRETIL on Kaggle (2 × T4)

ByT5-Sanskrit (`chronbmm/sanskrit5-multitask`, mode `SLM`: segmentation + lemma + morphosyntax;
Nehrdich, Hellwig & Keutzer 2024) over the GRETIL units from `scripts/prepare_gretil.py`.

* One model copy per GPU, each working on its own shards.
* Built for **Save Version → Save & Run All (Commit)**: runs with the browser closed and
  stops cleanly after `RUN_HOURS` (Kaggle ends sessions at 12 h).
* **Resuming:** attach the previous version's output as an input (see the instructions in the
  repo, `notebooks/KAGGLE.md`). Its annotations are copied in and skipped, so the newest
  version's output always holds *everything* annotated so far.

Settings needed: Accelerator **GPU T4 x2**, Internet **on**.

In [ ]:
RUN_HOURS = 11.0      # stop starting new batches after this many hours (session limit: 12 h)
BATCH = 128           # per GPU; halved automatically on out-of-memory
FORCE_FP32 = False    # True to skip the fp16 check and always use fp32

In [ ]:
import os, glob, time, json, shutil, subprocess, sys
T0 = time.time()
WORK = '/kaggle/working'
OUT = f'{WORK}/annotated'
os.makedirs(OUT, exist_ok=True)

shards = sorted(glob.glob('/kaggle/input/**/shard_*.txt', recursive=True))
assert len(shards) > 0, 'No shard_*.txt found: add the shards dataset as an input'
SHARD_DIR = os.path.dirname(shards[0])
print(len(shards), 'shards in', SHARD_DIR)

# carry over annotations from attached previous versions (any shard_*.tsv under /kaggle/input)
prev = sorted(glob.glob('/kaggle/input/**/shard_*.tsv', recursive=True))
carried = 0
for f in prev:
    dst = f'{OUT}/{os.path.basename(f)}'
    seen = set()
    if os.path.exists(dst):
        seen = {l.split('\t', 1)[0] for l in open(dst, encoding='utf-8') if '\t' in l}
    with open(dst, 'a', encoding='utf-8') as out:
        for l in open(f, encoding='utf-8'):
            k = l.split('\t', 1)[0]
            if '\t' in l and k not in seen:
                out.write(l if l.endswith('\n') else l + '\n'); seen.add(k); carried += 1
TOTAL = sum(sum(1 for _ in open(s, encoding='utf-8')) for s in shards)
print(f'carried over {carried:,} annotated units from {len(prev)} files; total units {TOTAL:,}')

import torch
NGPU = torch.cuda.device_count()
print('GPUs:', NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)])
assert NGPU > 0, 'No GPU: set Accelerator to GPU T4 x2'

In [ ]:
%%writefile /kaggle/working/worker.py
# One process per GPU (CUDA_VISIBLE_DEVICES set by the launcher).
import glob, os, random, sys, time
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer

gpu, ngpu, half = int(sys.argv[1]), int(sys.argv[2]), sys.argv[3] == '1'
deadline, shard_dir, out_dir, batch = float(sys.argv[4]), sys.argv[5], sys.argv[6], int(sys.argv[7])
MODEL, PREFIX, MAX_LEN = 'chronbmm/sanskrit5-multitask', 'SLM ', 512
log = open(f'{out_dir}/../worker{gpu}.log', 'a', buffering=1)

tok = AutoTokenizer.from_pretrained(MODEL)
model = T5ForConditionalGeneration.from_pretrained(MODEL).to('cuda').eval()
if half:
    model = model.half()

def analyse(lines):
    enc = tok([PREFIX + l for l in lines], return_tensors='pt', padding=True,
              truncation=True, max_length=MAX_LEN).to('cuda')
    cap = min(MAX_LEN, 4 * max(len(l.encode()) for l in lines) + 16)   # stop runaway outputs
    with torch.no_grad():
        out = model.generate(**enc, max_length=cap, num_beams=1)
    return tok.batch_decode(out, skip_special_tokens=True)

# fixed shuffled shard order (genres mixed if we stop early); this GPU takes every ngpu-th
order = sorted(glob.glob(f'{shard_dir}/shard_*.txt')); random.Random(0).shuffle(order)
for s in order[gpu::ngpu]:
    name = os.path.basename(s).replace('.txt', '.tsv')
    lines = open(s, encoding='utf-8').read().split('\n')
    if lines and lines[-1] == '':
        lines = lines[:-1]
    outp = f'{out_dir}/{name}'
    done = set()
    if os.path.exists(outp):
        done = {int(l.split('\t', 1)[0]) for l in open(outp, encoding='utf-8') if '\t' in l}
    todo = sorted((i for i in range(len(lines)) if i not in done), key=lambda i: len(lines[i]))
    if not todo:
        continue
    log.write(f'START {name} {len(todo)}\n')
    b, buf, k = 0, [], 0
    while b < len(todo):
        if time.time() > deadline:
            break
        idx = todo[b:b + batch]
        try:
            outs = analyse([lines[i] for i in idx])
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); batch = max(8, batch // 2); log.write(f'OOM batch={batch}\n'); continue
        buf += [f'{i}\t{o}\n' for i, o in zip(idx, outs)]
        b += len(idx); k += 1
        if k % 5 == 0 or b >= len(todo):
            with open(outp, 'a', encoding='utf-8') as f:
                f.writelines(buf)
            buf = []
            log.write(f'PROGRESS {name} {b}/{len(todo)}\n')
    if buf:
        with open(outp, 'a', encoding='utf-8') as f:
            f.writelines(buf)
    if time.time() > deadline:
        log.write('DEADLINE\n'); break
log.write('EXIT\n')

In [ ]:
# fp16 check: T5 models can overflow in half precision. Use fp16 only if it gives the same
# output as fp32 on (almost) all of 256 sample units; also measures speed.
from transformers import T5ForConditionalGeneration, AutoTokenizer
MODEL = 'chronbmm/sanskrit5-multitask'
tok = AutoTokenizer.from_pretrained(MODEL)
m = T5ForConditionalGeneration.from_pretrained(MODEL).to('cuda:0').eval()
sample = sorted(open(shards[0], encoding='utf-8').read().split('\n')[:256], key=len)
def run(model):
    t, outs = time.time(), []
    for i in range(0, len(sample), 128):
        lines = sample[i:i+128]
        enc = tok(['SLM ' + l for l in lines], return_tensors='pt', padding=True,
                  truncation=True, max_length=512).to('cuda:0')
        cap = min(512, 4 * max(len(l.encode()) for l in lines) + 16)
        with torch.no_grad():
            outs += tok.batch_decode(model.generate(**enc, max_length=cap, num_beams=1),
                                     skip_special_tokens=True)
    return outs, len(sample) / (time.time() - t)
o32, r32 = run(m)
o16, r16 = run(m.half())
same = sum(a == b for a, b in zip(o32, o16)) / len(sample)
HALF = (not FORCE_FP32) and same >= 0.99
rate = (r16 if HALF else r32) * NGPU
print(f'fp32 {r32:.1f} units/s, fp16 {r16:.1f} units/s per GPU; fp16 identical on {same:.1%}')
print(f'using {"fp16" if HALF else "fp32"}; ~{rate:.0f} units/s on {NGPU} GPU(s) '
      f'-> ~{TOTAL / rate / 3600:.0f} GPU-session hours for all {TOTAL:,} units')
print('example:', sample[-1], '\n ->', o32[-1])
del m; torch.cuda.empty_cache()

In [ ]:
deadline = T0 + RUN_HOURS * 3600
procs = [subprocess.Popen([sys.executable, f'{WORK}/worker.py', str(g), str(NGPU), str(int(HALF)),
                           str(deadline), SHARD_DIR, OUT, str(BATCH)],
                          env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(g)})
         for g in range(NGPU)]

def n_done():
    return sum(sum(1 for _ in open(f, encoding='utf-8')) for f in glob.glob(f'{OUT}/shard_*.tsv'))
start_done, t_start = n_done(), time.time()
while any(p.poll() is None for p in procs):
    time.sleep(600)
    d = n_done(); new = d - start_done; el = time.time() - t_start
    rate = new / el if el > 0 else 0
    eta = (TOTAL - d) / rate / 3600 if rate > 0 else float('nan')
    print(f'[{el/3600:4.1f} h] {d:,}/{TOTAL:,} ({100*d/TOTAL:.1f}%)  {rate:.1f} units/s  '
          f'remaining ~{eta:.0f} GPU-session h', flush=True)
print('workers exit codes:', [p.returncode for p in procs])
for g in range(NGPU):
    print(f'--- worker{g}.log (tail)'); print(''.join(open(f'{WORK}/worker{g}.log').readlines()[-3:]))

In [ ]:
d = n_done()
json.dump({'annotated_units': d, 'total_units': TOTAL, 'fp16': HALF,
           'finished_at_hours': (time.time() - T0) / 3600},
          open(f'{WORK}/progress.json', 'w'), indent=1)
print(f'{d:,}/{TOTAL:,} units annotated ({100*d/TOTAL:.1f}%)')
shutil.make_archive(f'{WORK}/annotated', 'zip', WORK, 'annotated')
print('annotated.zip written:', os.path.getsize(f'{WORK}/annotated.zip') // 2**20, 'MB')